### In this notebook we show how the model to model evaluation module works

In [12]:
# STENCIL DEFINITIONS
# ==============================================================================

# Updated based on your complete stencil list
START_EVENT_STENCILS = [
    "StartNoneEvent",
    "StartMessageEvent",
    "StartTimerEvent",
    "StartSignalEvent",
    "StartConditionalEvent",
    "StartErrorEvent",
    "StartEscalationEvent",
    "StartCompensationEvent",
    "StartMultipleEvent",
    "StartParallelMultipleEvent",
]

END_EVENT_STENCILS = [
    "EndNoneEvent",
    "EndMessageEvent",
    "EndTerminateEvent",
    "EndErrorEvent",
    "EndEscalationEvent",
    "EndCompensationEvent",
    "EndCancelEvent",
    "EndMultipleEvent",
    "EndSignalEvent",
]

INTERMEDIATE_EVENT_STENCILS = [
    "IntermediateEvent",
    "IntermediateMessageEventCatching",
    "IntermediateMessageEventThrowing",
    "IntermediateTimerEvent",
    "IntermediateErrorEvent",
    "IntermediateConditionalEvent",
    "IntermediateEscalationEvent",
    "IntermediateEscalationEventThrowing",
    "IntermediateSignalEventThrowing",
    "IntermediateSignalEventCatching",
    "IntermediateCompensationEventCatching",
    "IntermediateCompensationEventThrowing",
    "IntermediateCancelEvent",
    "IntermediateMultipleEventCatching",
    "IntermediateMultipleEventThrowing",
    "IntermediateParallelMultipleEventCatching",
    "IntermediateLinkEventThrowing",
    "IntermediateLinkEventCatching",
]

GATEWAY_STENCILS = [
    "Exclusive_Databased_Gateway",
    "ParallelGateway",
    "InclusiveGateway",
    "ComplexGateway",
    "EventbasedGateway",
]

ACTIVITY_STENCILS = ["Task", "Subprocess", "CollapsedSubprocess", "EventSubprocess", "CollapsedEventSubprocess"]

# All flow nodes (elements that can have sequence flows)
FLOW_NODE_STENCILS = (
    START_EVENT_STENCILS + END_EVENT_STENCILS + INTERMEDIATE_EVENT_STENCILS + GATEWAY_STENCILS + ACTIVITY_STENCILS
)

# Supporting elements
SUPPORTING_STENCILS = [
    "DataObject",
    "DataStore",
    "TextAnnotation",
    "Group",
    "Message",
    "Association_Unidirectional",
    "Association_Undirected",
    "Association_Bidirectional",
    "MessageFlow",
    "ITSystem",
]

from collections import Counter


def all_stencils(bpmn_models):
    all_stencils = []
    for i in final_count(bpmn_models):
        if i[0] not in all_stencils:
            all_stencils.append(i[0])
    return all_stencils


def count_stencil_ids(obj):
    """Recursively counts occurrences of all stencil IDs in a Signavio model tree."""
    counts = Counter()

    if isinstance(obj, dict):
        # If this object has a 'stencil' with an 'id', count it
        stencil = obj.get("stencil")
        if isinstance(stencil, dict) and "id" in stencil:
            counts[stencil["id"]] += 1
        # Recurse into any childShapes
        childshapes = obj.get("childShapes")
        if isinstance(childshapes, list):
            for child in childshapes:
                counts.update(count_stencil_ids(child))
        # Optionally, you can search other dict values in case the model is nonstandard
        # for v in obj.values():
        #    counts.update(count_stencil_ids(v))

    elif isinstance(obj, list):
        for item in obj:
            counts.update(count_stencil_ids(item))

    return counts


def final_count(bpmn_models):
    """
    Counts all stencil IDs across multiple BPMN models and returns a sorted list of counts.
    """

    final_dict = {}
    for i in bpmn_models:
        result = dict(count_stencil_ids(i))
        for m in result.keys():
            if m in final_dict:
                final_dict[m] += 1
            else:
                final_dict[m] = 1
    return sorted(final_dict.items(), key=lambda x: x[1], reverse=True)


def extract_elements_by_stencil_ids(model, stencil_ids):
    """
    Recursively extract all elements matching any of the given stencil IDs.

    Args:
        model: The BPMN model (JSON structure)
        stencil_ids: List of stencil IDs to match

    Returns:
        List of matching elements
    """
    results = []
    shapes = model.get("childShapes", [])
    for shape in shapes:
        if shape.get("stencil", {}).get("id") in stencil_ids:
            results.append(shape)
        if "childShapes" in shape:
            results.extend(extract_elements_by_stencil_ids(shape, stencil_ids))
    return results

In [13]:
import sys, json

sys.path.append("../")
sys.path.append("../model_evaluation/")

In [14]:
# load some examples from examples folder in Signavio json format
filename_ground_truth = f"../examples/E_j04.json"
with open(filename_ground_truth, "r") as infile:
    E4 = json.load(infile)


filename_generated = f"../examples/E_j04_4.bpmn2 _ Signavio.json"
with open(filename_generated, "r") as infile:
    E4_1 = json.load(infile)


filename_generated = f"../examples/process_complex.json"
with open(filename_generated, "r") as infile:
    pc = json.load(infile)

filename_generated = f"../examples/misc_booking_flight_tickets.json"
with open(filename_generated, "r") as infile:
    misc_loan_ft = json.load(infile)

filename_generated = f"../examples/misc_credit_quote_creation.json"
with open(filename_generated, "r") as infile:
    misc_loan_credit = json.load(infile)

filename_generated = f"../examples/Adrians_ex.json"
with open(filename_generated, "r") as infile:
    adrians_ex = json.load(infile)

filename_generated = f"../examples/simplemodel.json"
with open(filename_generated, "r") as infile:
    simple_model = json.load(infile)

In [ ]:
count_stencil_ids(misc_loan_ft)

In [15]:
from BPMN_conversion import BPMNConverter

# from bpmn_schema_helper import BPMNConverter as original_BPMNConverter

In [16]:
import json

# E4_json = BPMNConverter.convert(E4).to_json()

# E4_1_json = BPMNConverter.convert(E4_1).to_json()

misc_bft_json = BPMNConverter.convert(misc_loan_ft).to_json()
# misc_bft_json_original = original_BPMNConverter.convert(E4_1).to_json()

misc_credit_json = BPMNConverter.convert(misc_loan_credit).to_json()

pc_json = BPMNConverter.convert(pc).to_json()

# simple = BPMNConverter.convert(simple_model).to_json()
# pc_original_json = original_BPMNConverter.convert(pc).to_json()

# adrians_json = BPMNConverter.convert(adrians_ex).to_json()
# write E4_json to file
# with open("../E4_minimal.json", "w") as outfile:
#     E4_1_json = BPMNConverter.convert(misc_ft)
#     json.dump(json.loads(E4_1_json.to_json()), outfile, indent=4)

In [17]:
json.loads(misc_bft_json)
from bpmn_sets import extract_bpmn_sets

extract_bpmn_sets(json.loads(misc_bft_json))

{'activity_names': ['Plan travels',
  'Select the best offer and request tickets',
  'Create schedule',
  ''],
 'activity_types': ['Task', 'Send', 'Task', 'Subprocess'],
 'event_names': ['Feeling the Wanderlust',
  'Send travel request',
  'Get schedule',
  'Receive eTickets',
  'Travel can begin',
  'Receive confirmation',
  'travel request received',
  'Customer request processed',
  '2 min'],
 'event_types': ['StartNoneEvent',
  'IntermediateMessageEventThrowing',
  'IntermediateMessageEventCatching',
  'IntermediateMessageEventCatching',
  'EndNoneEvent',
  'IntermediateMessageEventCatching',
  'StartMessageEvent',
  'EndMessageEvent',
  'IntermediateTimerEvent'],
 'gateway_names': ['', '', ''],
 'gateway_types': ['Parallel', 'Parallel', 'Exclusive'],
 'seq_flows_str': ['Feeling the Wanderlust|Plan travels',
  'Plan travels|Send travel request',
  'Select the best offer and request tickets|Receive confirmation',
  'Receive confirmation|Create schedule',
  'Send travel request|Paral

## Complete BPMN Model Comparison Workflow

This section demonstrates the complete workflow for comparing two BPMN models using the evaluation framework.

In [ ]:
import sys
import json

sys.path.append("../model_evaluation/")

from BPMN_conversion import BPMNConverter
from bpmn_normalization import normalize_atomic_names
from string_similarity import bert_cosine_optimized
from bpmn_similarity import calculate_bpmn_similarity

# STEP 1: Load Signavio JSON files
print("Loading Signavio JSON files...")
with open("../examples/misc_booking_flight_tickets.json", "r") as f:
    ground_truth_signavio = json.load(f)

with open("../examples/test.json", "r") as f:
    generated_signavio = json.load(f)

# Convert to minimal BPMN format
print("\nConverting Signavio JSON to minimal BPMN format...")
ground_truth_minimal = BPMNConverter.convert(ground_truth_signavio)
generated_minimal = BPMNConverter.convert(generated_signavio)

# Parse to dict for normalization and comparison
ground_truth_dict = json.loads(ground_truth_minimal.to_json())
generated_dict = json.loads(generated_minimal.to_json())

print(
    f"  Ground Truth: {len(ground_truth_dict['activities'])} activities, "
    f"{len(ground_truth_dict['events'])} events, "
    f"{len(ground_truth_dict['gateways'])} gateways"
)
print(
    f"  Generated:    {len(generated_dict['activities'])} activities, "
    f"{len(generated_dict['events'])} events, "
    f"{len(generated_dict['gateways'])} gateways"
)

# STEP 3: Normalize atomic names (IMPORTANT: before similarity calculation!)
print("\nNormalizing atomic names using semantic similarity...")
print("  (This aligns element names like 'Book flight' ↔ 'Book a flight')")
generated_normalized, mappings = normalize_atomic_names(
    ground_truth_dict, generated_dict, bert_cosine_optimized, threshold=0.7
)

# if mappings:
#     print(f"  Applied {sum(len(v) for v in mappings.values())} name mappings:")
#     for atomic_type, mapping in mappings.items():
#         if mapping:
#             print(f"    {atomic_type}: {len(mapping)} mappings")
#             # Show first 2 examples
#             for i, (old_name, new_name) in enumerate(list(mapping.items())[:2]):
#                 print(f"      '{old_name}' → '{new_name}'")
# else:
#     print("  No semantic mappings needed (names already match)")

# STEP 4: Calculate similarity (AFTER normalization!)
print("\nCalculating BPMN similarity...")
similarity_results = calculate_bpmn_similarity(
    ground_truth_dict,
    generated_normalized,
    method="dice",  # Use normalized version!
)

print(f"\n{'='*60}")
print(f"OVERALL SIMILARITY: {similarity_results['overall']:.3f} ({similarity_results['overall']*100:.1f}%)")
print(f"{'='*60}")
print("\nHigh-Level Scores:")
for category, score in similarity_results["high_level_scores"].items():
    weight = similarity_results["weights_used"][category]
    print(f"  {category:20s} (weight={weight:4.0%}): {score:.3f} ({score*100:5.1f}%)")

Loading Signavio JSON files...

Converting Signavio JSON to minimal BPMN format...
  Ground Truth: 4 activities, 11 events, 3 gateways
  Generated:    3 activities, 8 events, 1 gateways

Normalizing atomic names using semantic similarity...
  (This aligns element names like 'Book flight' ↔ 'Book a flight')

Calculating BPMN similarity...

OVERALL SIMILARITY: 0.272 (27.2%)

High-Level Scores:
  structural           (weight= 30%): 0.474 ( 47.4%)
  flows                (weight= 50%): 0.145 ( 14.5%)
  organizational       (weight= 15%): 0.333 ( 33.3%)
  subprocess           (weight=  5%): 0.150 ( 15.0%)


### Comparison: With vs Without Normalization

This demonstrates the impact of semantic normalization on similarity scores.

In [22]:
# Compare similarity WITH normalization vs WITHOUT normalization
print("Comparing: WITH normalization vs WITHOUT normalization\n")

# WITHOUT normalization (direct comparison)
similarity_without_norm = calculate_bpmn_similarity(
    ground_truth_dict, generated_dict, method="dice"  # Original, not normalized
)

# WITH normalization (what we calculated above)
similarity_with_norm = similarity_results

print(f"{'Category':<20} {'Without Norm':>12} {'With Norm':>12} {'Improvement':>12}")
print("-" * 60)

for category in ["structural", "flows", "organizational", "subprocess"]:
    without = similarity_without_norm["high_level_scores"][category]
    with_norm = similarity_with_norm["high_level_scores"][category]
    improvement = with_norm - without
    print(f"{category:<20} {without:>12.3f} {with_norm:>12.3f} {improvement:>+12.3f}")

print("-" * 60)
overall_without = similarity_without_norm["overall"]
overall_with = similarity_with_norm["overall"]
overall_improvement = overall_with - overall_without

print(f"{'OVERALL':<20} {overall_without:>12.3f} {overall_with:>12.3f} {overall_improvement:>+12.3f}")
print(
    f"\nConclusion: Normalization {'improves' if overall_improvement > 0 else 'does not improve'} similarity by "
    f"{abs(overall_improvement)*100:.2f} percentage points"
)

Comparing: WITH normalization vs WITHOUT normalization

Category             Without Norm    With Norm  Improvement
------------------------------------------------------------
structural                  0.360        0.474       +0.114
flows                       0.000        0.145       +0.145
organizational              0.000        0.333       +0.333
subprocess                  0.000        0.150       +0.150
------------------------------------------------------------
OVERALL                     0.108        0.272       +0.165

Conclusion: Normalization improves similarity by 16.45 percentage points


In [ ]:
# ==============================================================================
# ALTERNATIVE: Quick comparison without normalization
# ==============================================================================
# Use this when you want a fast comparison and know names are already aligned,
# or when you're comparing against a reference without semantic variations.

# Load different models for this example
with open("../examples/E_j04.json", "r") as f:
    model1_signavio = json.load(f)

with open("../examples/E_j04_4.bpmn2 _ Signavio.json", "r") as f:
    model2_signavio = json.load(f)

# Convert to minimal BPMN
model1_dict = json.loads(BPMNConverter.convert(model1_signavio).to_json())
model2_dict = json.loads(BPMNConverter.convert(model2_signavio).to_json())

print("Model 1:")
print(f"  Activities: {len(model1_dict['activities'])}")
print(f"  Events: {len(model1_dict['events'])}")
print(f"  Gateways: {len(model1_dict['gateways'])}")

print("\nModel 2:")
print(f"  Activities: {len(model2_dict['activities'])}")
print(f"  Events: {len(model2_dict['events'])}")
print(f"  Gateways: {len(model2_dict['gateways'])}")

# Direct similarity calculation (no normalization)
quick_similarity = calculate_bpmn_similarity(model1_dict, model2_dict, method="dice")

print(f"\nOverall Similarity: {quick_similarity['overall']:.3f}")
print("\nNote: This skips normalization. Use the full pipeline above for better results.")

In [23]:
# ==============================================================================
# Using Different Similarity Metrics
# ==============================================================================
# The framework supports: "dice", "jaccard", "precision", "recall", "f1"

# Using the normalized models from above
print("Comparing different similarity metrics:\n")

methods = ["dice", "jaccard", "precision", "recall", "f1"]
results = {}

for method in methods:
    result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method=method)
    results[method] = result
    print(f"{method.upper():<12} Overall: {result['overall']:.3f}")

print("\n" + "=" * 60)
print("Metric Descriptions:")
print("  - Dice:      Balanced similarity (2*|A∩B| / |A|+|B|)")
print("  - Jaccard:   Set overlap (|A∩B| / |A∪B|)")
print("  - Precision: How much of generated is correct (|A∩B| / |B|)")
print("  - Recall:    How much of ground truth is captured (|A∩B| / |A|)")
print("  - F1:        Harmonic mean of precision and recall")

Comparing different similarity metrics:

DICE         Overall: 0.272
JACCARD      Overall: 0.183
PRECISION    Overall: 0.374
RECALL       Overall: 0.222
F1           Overall: 0.272

Metric Descriptions:
  - Dice:      Balanced similarity (2*|A∩B| / |A|+|B|)
  - Jaccard:   Set overlap (|A∩B| / |A∪B|)
  - Precision: How much of generated is correct (|A∩B| / |B|)
  - Recall:    How much of ground truth is captured (|A∩B| / |A|)
  - F1:        Harmonic mean of precision and recall


In [24]:
# ==============================================================================
# Using Custom Weights
# ==============================================================================
# Adjust importance of different BPMN aspects based on your evaluation needs

# Example 1: Flow-heavy weighting (control flow matters most)
flow_heavy_weights = {
    "structural": 0.20,  # 20% - element types and names
    "flows": 0.60,  # 60% - sequence and message flows
    "organizational": 0.15,  # 15% - pools and lanes
    "subprocess": 0.05,  # 5%  - subprocess structure
}

# Example 2: Structure-heavy weighting (elements matter most)
structure_heavy_weights = {"structural": 0.60, "flows": 0.25, "organizational": 0.10, "subprocess": 0.05}

# Compare different weightings
print("Comparing different weight configurations:\n")

configs = [("Default", None), ("Flow-Heavy", flow_heavy_weights), ("Structure-Heavy", structure_heavy_weights)]

for name, weights in configs:
    result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method="dice", weights=weights)
    print(f"{name:<20} Overall: {result['overall']:.3f}")
    if weights:
        print(
            f"  Weights used: structural={weights['structural']:.0%}, "
            f"flows={weights['flows']:.0%}, "
            f"org={weights['organizational']:.0%}, "
            f"subprocess={weights['subprocess']:.0%}"
        )
    print()

Comparing different weight configurations:

Default              Overall: 0.272

Flow-Heavy           Overall: 0.240
  Weights used: structural=20%, flows=60%, org=15%, subprocess=5%

Structure-Heavy      Overall: 0.361
  Weights used: structural=60%, flows=25%, org=10%, subprocess=5%



In [ ]:
# # ==============================================================================
# # SUMMARY: Best Practices for BPMN Comparison
# # ==============================================================================

# print(
#     """
# ╔═══════════════════════════════════════════════════════════════════════════╗
# ║                   RECOMMENDED BPMN COMPARISON PIPELINE                    ║
# ╠═══════════════════════════════════════════════════════════════════════════╣
# ║                                                                           ║
# ║  1. Load Signavio JSON                                                    ║
# ║     └─ json.load() from file                                             ║
# ║                                                                           ║
# ║  2. Convert to Minimal BPMN (BPMN_conversion.py)                         ║
# ║     └─ BPMNConverter.convert(signavio_json)                              ║
# ║     └─ Normalizes structure, handles subprocesses                        ║
# ║                                                                           ║
# ║  3. Normalize Atomic Names (bpmn_normalization.py) *** IMPORTANT ***     ║
# ║     └─ normalize_atomic_names(model1, model2, bert_cosine_optimized)    ║
# ║     └─ Aligns semantic variations in element names                       ║
# ║     └─ Example: "Book flight" ↔ "Book a flight"                         ║
# ║                                                                           ║
# ║  4. Extract BPMN Sets (bpmn_sets.py) - happens automatically             ║
# ║     └─ extract_bpmn_sets() called internally                             ║
# ║     └─ Separates top-level from subprocess elements                      ║
# ║                                                                           ║
# ║  5. Calculate Similarity (bpmn_similarity.py)                            ║
# ║     └─ calculate_bpmn_similarity(model1, model2_normalized, method)     ║
# ║     └─ Returns fine/grouped/high-level scores + overall                  ║
# ========================================================================
# """
# )

In [60]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np

print("Calculating similarity scores (this may take a moment)...")

# Cache for results with different metrics and thresholds
results_cache = {}
current_metric = "dice"
current_threshold = 0.7
current_normalized = generated_normalized

# Initial calculation
base_result = calculate_bpmn_similarity(ground_truth_dict, generated_normalized, method="dice")
results_cache[("dice", 0.7)] = (generated_normalized, base_result)

has_subprocess = base_result.get("has_expanded_subprocess", False)

# Get the default weights that were actually used in the calculation
default_weights = base_result["weights_used"]

output = widgets.Output()

# Use 0-100 range for easier percentage input
structural_slider = widgets.FloatSlider(
    value=default_weights["structural"] * 100,
    min=0,
    max=100,
    step=1,
    description="Structural (%):",
    continuous_update=False,
)
flows_slider = widgets.FloatSlider(
    value=default_weights["flows"] * 100, min=0, max=100, step=1, description="Flows (%):", continuous_update=False
)
organizational_slider = widgets.FloatSlider(
    value=default_weights["organizational"] * 100,
    min=0,
    max=100,
    step=1,
    description="Org (%):",
    continuous_update=False,
)
subprocess_slider = widgets.FloatSlider(
    value=default_weights["subprocess"] * 100,
    min=0,
    max=100 if has_subprocess else 0,
    step=1,
    description="Subprocess (%):",
    continuous_update=False,
    disabled=not has_subprocess,
)

# String similarity threshold slider
threshold_slider = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Threshold:",
    continuous_update=False,
    tooltip="String similarity threshold for normalization (0.7 = 70% match required)",
)


def normalize_weights(structural, flows, organizational, subprocess):
    if not has_subprocess:
        subprocess = 0.0
    total = structural + flows + organizational + subprocess
    if total == 0:
        return 0.35, 0.45, 0.15, 0.05
    return structural / total, flows / total, organizational / total, subprocess / total


def get_result_for_metric_and_threshold(metric, threshold):
    """Get or calculate result for given metric and threshold"""
    global results_cache, current_normalized

    cache_key = (metric, threshold)

    if cache_key not in results_cache:
        # Need to recalculate with new threshold
        if threshold != 0.7:
            print(f"Recalculating with threshold {threshold:.2f}...")
            normalized, mappings = normalize_atomic_names(
                ground_truth_dict, generated_dict, bert_cosine_optimized, threshold=threshold
            )
        else:
            normalized = generated_normalized

        result = calculate_bpmn_similarity(ground_truth_dict, normalized, method=metric)
        results_cache[cache_key] = (normalized, result)

    return results_cache[cache_key]


def update_visualization(structural, flows, organizational, subprocess):
    global current_metric, current_threshold, current_normalized

    structural, flows, organizational, subprocess = normalize_weights(structural, flows, organizational, subprocess)

    weights = {"structural": structural, "flows": flows, "organizational": organizational, "subprocess": subprocess}

    # Get the result for current metric and threshold
    current_normalized, result = get_result_for_metric_and_threshold(current_metric, current_threshold)

    overall = (
        result["high_level_scores"]["structural"] * structural
        + result["high_level_scores"]["flows"] * flows
        + result["high_level_scores"]["organizational"] * organizational
        + result["high_level_scores"]["subprocess"] * subprocess
    )

    with output:
        clear_output(wait=True)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        categories = ["Structural", "Flows", "Organizational", "Subprocess"]
        scores = [
            result["high_level_scores"]["structural"],
            result["high_level_scores"]["flows"],
            result["high_level_scores"]["organizational"],
            result["high_level_scores"]["subprocess"],
        ]

        # Calculate equal weights (baseline comparison)
        if has_subprocess:
            equal_weight = 0.25  # 25% each when subprocess is present
        else:
            equal_weight = 1.0 / 3.0  # 33.33% each when no subprocess

        equal_weights_vals = [equal_weight, equal_weight, equal_weight, equal_weight if has_subprocess else 0]
        equal_weighted_scores = [s * w for s, w in zip(scores, equal_weights_vals)]

        # Calculate current (custom) weights
        weights_vals = [structural, flows, organizational, subprocess]
        weighted_scores = [s * w for s, w in zip(scores, weights_vals)]

        # Calculate overall scores for both
        overall_equal = sum(equal_weighted_scores)

        colors = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
        y_pos = np.arange(len(categories))

        bars = ax1.barh(y_pos, equal_weighted_scores, color=colors, alpha=0.5, label="Equal Weights")
        bars_w = ax1.barh(y_pos, weighted_scores, color=colors, alpha=0.9, label="Current Weights")

        ax1.set_yticks(y_pos)
        ax1.set_yticklabels(categories)
        ax1.set_xlabel("Score")
        ax1.set_xlim(0, 1)
        metric_display = current_metric.upper()
        ax1.set_title(
            f"Equal={overall_equal:.1%} | Metric: {metric_display} | Threshold: {current_threshold:.2f}",
            fontsize=11,
        )
        ax1.legend(loc="lower right")
        ax1.grid(axis="x", alpha=0.3)

        for i, (equal_score, weighted) in enumerate(zip(equal_weighted_scores, weighted_scores)):
            if equal_score > 0.01:
                ax1.text(equal_score + 0.01, i + 0.15, f"{equal_score:.2f}", va="center", fontsize=8, color="gray")
            if weighted > 0.01:
                ax1.text(weighted + 0.01, i - 0.15, f"{weighted:.2f}", va="center", fontsize=8, weight="bold")

        # Display overall similarity prominently in the center
        ax1.text(
            0.5,
            0.5,
            f"{overall:.1%}",
            transform=ax1.transAxes,
            fontsize=48,
            weight="bold",
            ha="center",
            va="center",
            alpha=0.15,
            color="black",
        )

        # Right: Fine-grained element breakdown
        elements = [
            "Activities",
            "Events",
            "Gateways",
            "Seq Flows",
            "Msg Flows",
            "Lanes",
            "Subprocess\nNames",
            "Subprocess\nElements",
            "Subprocess\nFlows",
        ]
        element_scores = [
            result["activity_names"],
            result["event_names"],
            result["gateway_names"],
            result["seq_flows_str"],
            result["mes_flows_str"],
            result["lane_names"],
            result["subprocess_names"],
            result["subprocess_elemrefs"],
            result["subprocess_flows"],
        ]
        element_colors = ["#3498db"] * 3 + ["#2ecc71"] * 2 + ["#f39c12"] + ["#9b59b6"] * 3

        y_pos2 = np.arange(len(elements))
        bars2 = ax2.barh(y_pos2, element_scores, color=element_colors, alpha=0.8)

        ax2.set_yticks(y_pos2)
        ax2.set_yticklabels(elements, fontsize=9)
        ax2.set_xlabel("Score")
        ax2.set_xlim(0, 1)
        ax2.set_title("Element-Level Breakdown", fontsize=13)
        ax2.grid(axis="x", alpha=0.3)

        for i, score in enumerate(element_scores):
            ax2.text(score + 0.02, i, f"{score:.2f}", va="center", fontsize=8)

        plt.tight_layout()
        plt.show()


# Create metric selection buttons
dice_button = widgets.Button(description="Dice", button_style="primary", tooltip="Dice coefficient (default)")
jaccard_button = widgets.Button(description="Jaccard", button_style="", tooltip="Jaccard similarity")
precision_button = widgets.Button(description="Precision", button_style="", tooltip="Precision score")
recall_button = widgets.Button(description="Recall", button_style="", tooltip="Recall score")
f1_button = widgets.Button(description="F1", button_style="", tooltip="F1 score")

# Create control buttons
recalculate_button = widgets.Button(
    description="Recalculate",
    button_style="success",
    icon="refresh",
    tooltip="Apply current weights and update visualization",
)

reset_button = widgets.Button(
    description="Reset to Default",
    button_style="info",
    icon="undo",
    tooltip=f"Reset to default weights: S={default_weights['structural']:.0%}, F={default_weights['flows']:.0%}, O={default_weights['organizational']:.0%}, Sub={default_weights['subprocess']:.0%}",
)

message_output = widgets.Output()


def set_metric(metric_name):
    """Change the active metric and update button styles"""
    global current_metric
    current_metric = metric_name

    # Update button styles
    dice_button.button_style = "primary" if metric_name == "dice" else ""
    jaccard_button.button_style = "primary" if metric_name == "jaccard" else ""
    precision_button.button_style = "primary" if metric_name == "precision" else ""
    recall_button.button_style = "primary" if metric_name == "recall" else ""
    f1_button.button_style = "primary" if metric_name == "f1" else ""

    # Recalculate with current weights
    recalculate_with_weights(None)


def on_threshold_change(change):
    """Handle threshold slider changes"""
    global current_threshold
    current_threshold = change["new"]


def recalculate_with_weights(button):
    """Update the visualization when recalculate button is clicked"""
    # Convert from percentage (0-100) to proportion (0-1)
    structural = structural_slider.value / 100.0
    flows = flows_slider.value / 100.0
    organizational = organizational_slider.value / 100.0
    subprocess = subprocess_slider.value / 100.0

    if not has_subprocess:
        subprocess = 0.0

    # Calculate total
    total = structural + flows + organizational + subprocess

    # Normalize if needed
    normalized = False
    if abs(total - 1.0) > 0.01:  # Use slightly larger tolerance for percentage rounding
        if total == 0:
            structural = default_weights["structural"]
            flows = default_weights["flows"]
            organizational = default_weights["organizational"]
            subprocess = default_weights["subprocess"]
        else:
            structural = structural / total
            flows = flows / total
            organizational = organizational / total
            subprocess = subprocess / total
        normalized = True

        # Update sliders to show normalized values (convert back to percentage)
        structural_slider.value = structural * 100
        flows_slider.value = flows * 100
        organizational_slider.value = organizational * 100
        subprocess_slider.value = subprocess * 100

    # Show normalization message if needed
    with message_output:
        clear_output(wait=True)
        if normalized:
            print("⚠️ Weights were normalized to sum to 100%")

    # Call the update function with normalized weights (in proportion form)
    update_visualization(structural, flows, organizational, subprocess)


def reset_to_defaults(button):
    """Reset sliders to default weights and recalculate"""
    # Convert default weights to percentage for sliders
    structural_slider.value = default_weights["structural"] * 100
    flows_slider.value = default_weights["flows"] * 100
    organizational_slider.value = default_weights["organizational"] * 100
    subprocess_slider.value = default_weights["subprocess"] * 100

    with message_output:
        clear_output(wait=True)
        print("✓ Reset to default weights")

    # Recalculate with default weights (in proportion form)
    update_visualization(
        default_weights["structural"],
        default_weights["flows"],
        default_weights["organizational"],
        default_weights["subprocess"],
    )


# Connect buttons to functions
dice_button.on_click(lambda b: set_metric("dice"))
jaccard_button.on_click(lambda b: set_metric("jaccard"))
precision_button.on_click(lambda b: set_metric("precision"))
recall_button.on_click(lambda b: set_metric("recall"))
f1_button.on_click(lambda b: set_metric("f1"))
recalculate_button.on_click(recalculate_with_weights)
reset_button.on_click(reset_to_defaults)
threshold_slider.observe(on_threshold_change, names="value")

# Display initial visualization with default weights (in proportion form)
update_visualization(
    default_weights["structural"],
    default_weights["flows"],
    default_weights["organizational"],
    default_weights["subprocess"],
)

# Layout the widgets
display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Adjust Similarity Weights</h3>"),
            widgets.HTML("<p style='color: gray; font-size: 11px;'>Select metric and adjust parameters</p>"),
            widgets.HTML("<b>Similarity Metric:</b>"),
            widgets.HBox([dice_button, jaccard_button, precision_button, recall_button, f1_button]),
            widgets.HTML("<b>Normalization Threshold:</b>"),
            threshold_slider,
            widgets.HTML("<b>Category Weights (should sum to 100%):</b>"),
            structural_slider,
            flows_slider,
            organizational_slider,
            subprocess_slider,
            widgets.HBox([recalculate_button, reset_button]),
            message_output,
            output,
        ]
    )
)

Calculating similarity scores (this may take a moment)...


In [52]:
base_result

{'fine_scores': {'activity_names': 0.2857142857142857,
  'activity_types': 0.8571428571428571,
  'event_names': 0.4,
  'event_types': 0.8,
  'gateway_names': 0.0,
  'gateway_types': 0.5,
  'seq_flows_str': 0.09090909090909091,
  'mes_flows_str': 0.2,
  'lane_names': 0.3333333333333333,
  'lane_with_refs': 0.3333333333333333,
  'subprocess_names': 0.0,
  'subprocess_elemrefs': 0.5,
  'subprocess_flows': 0.0},
 'grouped_scores': {'activities': 0.5714285714285714,
  'events': 0.6000000000000001,
  'gateways': 0.25,
  'flows': 0.14545454545454545,
  'pools': 0.3333333333333333,
  'subprocess': 0.15},
 'high_level_scores': {'structural': 0.47380952380952385,
  'flows': 0.14545454545454545,
  'organizational': 0.3333333333333333,
  'subprocess': 0.15},
 'overall': 0.2723701298701299,
 'weights_used': {'structural': 0.3,
  'flows': 0.5,
  'organizational': 0.15,
  'subprocess': 0.05},
 'has_expanded_subprocess': True,
 'activity_names': 0.2857142857142857,
 'activity_types': 0.857142857142857